# 02 — Nanoparticle Feature Engineering

## Objective

Transform nanoparticle formulation variables and molecular reference structures into numerical features suitable for UMAP visualization and subsequent machine-learning analyses.

## Main tasks

1. Load the formulation dataset and molecular reference profiles.
2. Encode categorical and numerical formulation variables.
3. Calculate RDKit molecular descriptors for carrier and β-glucan reference fragments.
4. Integrate material properties and molecular descriptors into formulation-level feature tables.
5. Validate and export feature matrices for downstream analysis.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

pd.set_option("display.max_columns", 100)


## Dataset and molecular reference profiles

The formulation table contains design variables and experimental responses. The carrier profile provides representative SMILES for five base carriers, six amine-functionalized carriers, and three β-glucans. Molecular structures are reference fragments rather than complete polymer chains.

In [2]:
candidates = [Path("Dataset.xlsx"), Path("../data/raw/Dataset.xlsx"), Path("data/raw/Dataset.xlsx"), Path("/mnt/data/Dataset.xlsx")]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place Dataset.xlsx beside the notebook or in ../data/raw/")

OUTPUT_DIR = Path("../data/processed") if Path("../data").exists() else Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(DATA_PATH, sheet_name="Nanoparticle Formulation Table")
df.columns = df.columns.str.strip()
raw_profile = pd.read_excel(DATA_PATH, sheet_name="Drug Carriers Profile", header=None)
profile = raw_profile.iloc[:, 1:5].copy()
profile.columns = ["Material_ID", "Chemical_Name", "Material_Name", "SMILES"]
profile["Material_ID"] = profile["Material_ID"].astype(str).str.strip()
profile = profile[profile["Material_ID"].str.fullmatch(r"D[1-5]|N[1-6]|YG|BG|C")].copy()
profile["SMILES"] = profile["SMILES"].astype("string").str.replace("\\", "", regex=False).str.strip()
profile = profile.drop_duplicates("Material_ID").reset_index(drop=True)
print("Formulations:", df.shape, "| Molecular profiles:", len(profile))
display(profile[["Material_ID", "Material_Name", "SMILES"]])

Formulations: (6575, 16) | Molecular profiles: 14


,Material_ID,Material_Name,SMILES
0,D1,DSPE-mPEG,CCCCCCCCCCCCCCCCCC(=O)OCC(COP(=O)([O-])OCCNC(=...
1,D2,PLGA-mPEG,COCCOC(=O)COC(=O)C(C)O
2,D3,PLA-mPEG,COCCOC(=O)C(C)O
3,D4,PCL-mPEG,COCCOC(=O)CCCCCO
4,D5,Pluronic® F-127,OCCOCC(C)OCCO
5,N1,DSPE-PEG-NH2,CCCCCCCCCCCCCCCCCC(=O)OCC(COP(=O)(O)OCCNC(=O)O...
6,N2,PLGA-PEG-NH2,NCCOC(=O)COC(=O)C(C)O
7,N3,PLA-PEG-NH2,NCCOC(=O)C(C)O
8,N4,PCL-PEG-NH2,NCCOC(=O)CCCCCO
9,N5,Pluronic® F-127-NH2,NCCOCC(C)OCCO


## Formulation variable encoding

Categorical design variables are represented using one-hot encoding. Numerical ratios are standardized using StandardScaler so that differences in feature scale do not dominate distance-based analyses. Missing categorical values are explicitly labeled; numeric imputation is limited to design variables and does not alter experimental outcomes.

In [3]:
categorical_features = ["Glucan Type", "Modification", "Carrier 1", "Carrier 2 (NH2)", "β-Glucan MW"]
numeric_features = ["Carrier 1 Mass Ratio", "Carrier 2 Mass Ratio", "β-Glucan / Carrier 2 Mass Ratio"]
assert df["ID"].is_unique, "Formulation IDs must be unique"
assert set(categorical_features + numeric_features).issubset(df.columns)
X = df[categorical_features + numeric_features].copy()
X[categorical_features] = X[categorical_features].fillna("Not applicable").astype(str)
X[numeric_features] = X[numeric_features].apply(pd.to_numeric, errors="coerce")
print("Missing design values:")
display(X.isna().sum().to_frame("Missing"))
# Numeric missingness is retained as a separate indicator before median imputation.
missing_indicators = X[numeric_features].isna().astype(int).add_prefix("missing__")
for col in numeric_features:
    median = X[col].median()
    X[col] = X[col].fillna(median if pd.notna(median) else 0.0)
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded = encoder.fit_transform(X[categorical_features])
cat_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(categorical_features), index=df.index).add_prefix("cat__")
scaler = StandardScaler()
num_df = pd.DataFrame(scaler.fit_transform(X[numeric_features]), columns=["num__" + c for c in numeric_features], index=df.index)
formulation_features = pd.concat([cat_df, num_df, missing_indicators], axis=1)
print("Formulation feature matrix:", formulation_features.shape)
display(formulation_features.head())

Missing design values:


,Missing
Glucan Type,0
Modification,0
Carrier 1,0
Carrier 2 (NH2),0
β-Glucan MW,0
Carrier 1 Mass Ratio,0
Carrier 2 Mass Ratio,0
β-Glucan / Carrier 2 Mass Ratio,95


Formulation feature matrix: (6575, 30)


,cat__Glucan Type_Barley Glucan,cat__Glucan Type_Blank NP,cat__Glucan Type_Curdlan,cat__Glucan Type_Yeast Glucan,cat__Modification_H₂O₂ oxidation,cat__Modification_Not applicable,cat__Modification_Succinic anhydride modification,cat__Carrier 1_D1,cat__Carrier 1_D2,cat__Carrier 1_D3,cat__Carrier 1_D4,cat__Carrier 1_D5,cat__Carrier 2 (NH2)_N1,cat__Carrier 2 (NH2)_N2,cat__Carrier 2 (NH2)_N3,cat__Carrier 2 (NH2)_N4,cat__Carrier 2 (NH2)_N5,cat__Carrier 2 (NH2)_N6,cat__Carrier 2 (NH2)_Not applicable,cat__β-Glucan MW_High,cat__β-Glucan MW_Low,cat__β-Glucan MW_Medium,cat__β-Glucan MW_Not applicable,cat__β-Glucan MW_Unspecified,num__Carrier 1 Mass Ratio,num__Carrier 2 Mass Ratio,num__β-Glucan / Carrier 2 Mass Ratio,missing__Carrier 1 Mass Ratio,missing__Carrier 2 Mass Ratio,missing__β-Glucan / Carrier 2 Mass Ratio
0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.222246,-1.222246,-1.109249,0,0,0
1,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.222246,-1.222246,-0.210821,0,0,0
2,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.222246,-1.222246,1.329342,0,0,0
3,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.222246,-1.222246,-1.109249,0,0,0
4,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.222246,-1.222246,-0.210821,0,0,0


## Molecular descriptors

RDKit descriptors summarize selected structural properties, including fragment molecular weight, lipophilicity, topological polar surface area, hydrogen-bond donors and acceptors, and rotatable bonds. These descriptors describe the selected SMILES fragments and must not be interpreted as measured molecular weights or complete polymer properties.

In [4]:
profile["Molecule"] = profile["SMILES"].apply(lambda s: Chem.MolFromSmiles(str(s)) if pd.notna(s) else None)
profile["SMILES_Valid"] = profile["Molecule"].notna()
print("Valid SMILES:", profile["SMILES_Valid"].sum(), "/", len(profile))
display(profile[["Material_ID", "SMILES_Valid"]])

def molecular_descriptors(mol):
    keys = ["Fragment_MolWt", "LogP", "TPSA", "HBD", "HBA", "RotatableBonds"]
    if mol is None:
        return pd.Series({k: np.nan for k in keys})
    return pd.Series({
        "Fragment_MolWt": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": rdMolDescriptors.CalcNumHBD(mol),
        "HBA": rdMolDescriptors.CalcNumHBA(mol),
        "RotatableBonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
    })

profile = pd.concat([profile, profile["Molecule"].apply(molecular_descriptors)], axis=1)
descriptor_cols = ["Fragment_MolWt", "LogP", "TPSA", "HBD", "HBA", "RotatableBonds"]
display(profile[["Material_ID", "SMILES_Valid"] + descriptor_cols])

Valid SMILES: 14 / 14


,Material_ID,SMILES_Valid
0,D1,True
1,D2,True
2,D3,True
3,D4,True
4,D5,True
5,N1,True
6,N2,True
7,N3,True
8,N4,True
9,N5,True


,Material_ID,SMILES_Valid,Fragment_MolWt,LogP,TPSA,HBD,HBA,RotatableBonds
0,D1,True,872.151,8.8426,158.75,1.0,11.0,45.0
1,D2,True,206.194,-0.9000,82.06,1.0,6.0,6.0
2,D3,True,148.158,-0.4432,55.76,1.0,4.0,4.0
3,D4,True,190.239,0.7287,55.76,1.0,4.0,8.0
4,D5,True,164.201,-0.6073,58.92,2.0,4.0,7.0
5,N1,True,896.242,11.9615,216.94,4.0,12.0,47.0
6,N2,True,191.183,-1.5877,98.85,2.0,6.0,5.0
7,N3,True,133.147,-1.1309,72.55,2.0,4.0,3.0
8,N4,True,175.228,0.0410,72.55,2.0,4.0,7.0
9,N5,True,163.217,-0.6409,64.71,2.0,4.0,7.0


## Material properties and carrier classification

Material identity is retained alongside molecular descriptors. Carrier family and nominal polymer characteristics provide additional context for polymeric and lipid-based nanoparticle formulations. Nominal values below are modeling assumptions and should be adjusted to match actual material specifications.

In [5]:
family = {"D1":"Lipid", "D2":"PLGA", "D3":"PLA", "D4":"PCL", "D5":"Poloxamer", "N1":"Lipid", "N2":"PLGA", "N3":"PLA", "N4":"PCL", "N5":"Poloxamer", "N6":"Lipid"}
# Nominal PEG/polymer molecular weights (Da); unknown values remain missing.
nominal_polymer_mw = {"D2":10000, "D3":10000, "D4":10000, "N2":10000, "N3":10000, "N4":10000}
carrier_profile = profile[profile["Material_ID"].str.match(r"[DN]\d+")].copy()
carrier_profile["Carrier_Family"] = carrier_profile["Material_ID"].map(family)
carrier_profile["PEG_MW"] = carrier_profile["Material_ID"].map(lambda x: 2000 if x in {"D1", "N1", "N6", "D2", "D3", "D4", "N2", "N3", "N4"} else np.nan)
carrier_profile["Polymer_MW"] = carrier_profile["Material_ID"].map(nominal_polymer_mw)
carrier_profile["Amine_Functionalized"] = carrier_profile["Material_ID"].str.startswith("N").astype(int)
family_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
family_matrix = family_encoder.fit_transform(carrier_profile[["Carrier_Family"]])
family_df = pd.DataFrame(family_matrix, columns=["Family_" + s for s in family_encoder.categories_[0]], index=carrier_profile.index)
carrier_profile = pd.concat([carrier_profile, family_df], axis=1)
carrier_property_cols = descriptor_cols + ["PEG_MW", "Polymer_MW", "Amine_Functionalized"] + list(family_df.columns)
display(carrier_profile[["Material_ID", "Carrier_Family"] + carrier_property_cols])

,Material_ID,Carrier_Family,Fragment_MolWt,LogP,TPSA,HBD,HBA,RotatableBonds,PEG_MW,Polymer_MW,Amine_Functionalized,Family_Lipid,Family_PCL,Family_PLA,Family_PLGA,Family_Poloxamer
0,D1,Lipid,872.151,8.8426,158.75,1.0,11.0,45.0,2000.0,NaN,0,1.0,0.0,0.0,0.0,0.0
1,D2,PLGA,206.194,-0.9000,82.06,1.0,6.0,6.0,2000.0,10000.0,0,0.0,0.0,0.0,1.0,0.0
2,D3,PLA,148.158,-0.4432,55.76,1.0,4.0,4.0,2000.0,10000.0,0,0.0,0.0,1.0,0.0,0.0
3,D4,PCL,190.239,0.7287,55.76,1.0,4.0,8.0,2000.0,10000.0,0,0.0,1.0,0.0,0.0,0.0
4,D5,Poloxamer,164.201,-0.6073,58.92,2.0,4.0,7.0,NaN,NaN,0,0.0,0.0,0.0,0.0,1.0
5,N1,Lipid,896.242,11.9615,216.94,4.0,12.0,47.0,2000.0,NaN,1,1.0,0.0,0.0,0.0,0.0
6,N2,PLGA,191.183,-1.5877,98.85,2.0,6.0,5.0,2000.0,10000.0,1,0.0,0.0,0.0,1.0,0.0
7,N3,PLA,133.147,-1.1309,72.55,2.0,4.0,3.0,2000.0,10000.0,1,0.0,0.0,1.0,0.0,0.0
8,N4,PCL,175.228,0.0410,72.55,2.0,4.0,7.0,2000.0,10000.0,1,0.0,1.0,0.0,0.0,0.0
9,N5,Poloxamer,163.217,-0.6409,64.71,2.0,4.0,7.0,NaN,NaN,1,0.0,0.0,0.0,0.0,1.0


## Integrate carrier and β-glucan features

The same descriptor definitions are applied to both carrier positions and to the β-glucan reference structure. Position-specific prefixes distinguish the base carrier (`C1_`), amine-functionalized carrier (`C2_`), and β-glucan (`G_`) contributions.

In [6]:
glucan_map = {"Yeast Glucan":"YG", "Barley Glucan":"BG", "Curdlan":"C", "YG":"YG", "BG":"BG", "C":"C"}
working = df[["ID", "Glucan Type", "Carrier 1", "Carrier 2 (NH2)"]].copy()
working["Glucan_ID"] = working["Glucan Type"].map(glucan_map)

def property_table(source, id_col, target_col, prefix, cols):
    out = source[[id_col] + cols].copy()
    return out.rename(columns={id_col:target_col, **{c:prefix+c for c in cols}})

c1 = property_table(carrier_profile, "Material_ID", "Carrier 1", "C1_", carrier_property_cols)
c2 = property_table(carrier_profile, "Material_ID", "Carrier 2 (NH2)", "C2_", carrier_property_cols)
glucan_profile = profile[profile["Material_ID"].isin(["YG", "BG", "C"])].copy()
g = property_table(glucan_profile, "Material_ID", "Glucan_ID", "G_", descriptor_cols)
working = working.merge(c1, on="Carrier 1", how="left", validate="many_to_one")
working = working.merge(c2, on="Carrier 2 (NH2)", how="left", validate="many_to_one")
working = working.merge(g, on="Glucan_ID", how="left", validate="many_to_one")
print("Merged rows:", len(working))
assert len(working) == len(df)
for key, col in [("Carrier 1", "C1_Fragment_MolWt"), ("Carrier 2 (NH2)", "C2_Fragment_MolWt"), ("Glucan_ID", "G_Fragment_MolWt")]:
    bad = working[key].notna() & working[col].isna()
    print("Unmatched", key, ":", bad.sum())
    if bad.any():
        print(working.loc[bad, key].value_counts().head())

Merged rows: 6575
Unmatched Carrier 1 : 0
Unmatched Carrier 2 (NH2) : 0
Unmatched Glucan_ID : 0


## Standardize molecular and material features

Continuous material descriptors are standardized for numerical analysis. Binary material-family indicators remain 0/1. Missing properties are represented with missingness indicators and imputed for model compatibility; they must not be interpreted as experimentally measured zero values.

In [7]:
property_features = working[[c for c in working.columns if c.startswith(("C1_", "C2_", "G_"))]].copy()
continuous_names = descriptor_cols + ["PEG_MW", "Polymer_MW"]
continuous_cols = [p + c for p in ("C1_", "C2_", "G_") for c in continuous_names if p+c in property_features]
for col in continuous_cols:
    property_features[col] = pd.to_numeric(property_features[col], errors="coerce")
missing_property = property_features[continuous_cols].isna().astype(int).add_prefix("missing__")
property_features[continuous_cols] = property_features[continuous_cols].fillna(property_features[continuous_cols].median()).fillna(0)
property_features[continuous_cols] = StandardScaler().fit_transform(property_features[continuous_cols])
property_features = pd.concat([property_features.fillna(0), missing_property], axis=1)
assert not property_features.isna().any().any()
print("Property features:", property_features.shape)

Property features: (6575, 56)


## Feature matrices for downstream analysis

Two complementary representations are constructed. The full feature set combines formulation variables with molecular/material properties. The property-based feature set excludes explicit carrier identity one-hot columns, allowing analysis of relationships based more strongly on physicochemical descriptors. Glucan source identity is retained as a separate design variable in both representations.

In [8]:
full_features = pd.concat([formulation_features, property_features], axis=1)
carrier_identity_cols = [c for c in formulation_features.columns if c.startswith("cat__Carrier 1_") or c.startswith("cat__Carrier 2 (NH2)_")]
property_based_features = full_features.drop(columns=carrier_identity_cols)
full_df = pd.concat([df[["ID"]], full_features], axis=1)
property_df = pd.concat([df[["ID"]], property_based_features], axis=1)
assert full_df["ID"].is_unique and property_df["ID"].is_unique
assert not full_df.drop(columns="ID").isna().any().any()
assert not property_df.drop(columns="ID").isna().any().any()
print("Full feature set:", full_df.shape)
print("Property-based feature set:", property_df.shape)
display(full_df.head())

Full feature set: (6575, 87)
Property-based feature set: (6575, 75)


,ID,cat__Glucan Type_Barley Glucan,cat__Glucan Type_Blank NP,cat__Glucan Type_Curdlan,cat__Glucan Type_Yeast Glucan,cat__Modification_H₂O₂ oxidation,cat__Modification_Not applicable,cat__Modification_Succinic anhydride modification,cat__Carrier 1_D1,cat__Carrier 1_D2,cat__Carrier 1_D3,cat__Carrier 1_D4,cat__Carrier 1_D5,cat__Carrier 2 (NH2)_N1,cat__Carrier 2 (NH2)_N2,cat__Carrier 2 (NH2)_N3,cat__Carrier 2 (NH2)_N4,cat__Carrier 2 (NH2)_N5,cat__Carrier 2 (NH2)_N6,cat__Carrier 2 (NH2)_Not applicable,cat__β-Glucan MW_High,cat__β-Glucan MW_Low,cat__β-Glucan MW_Medium,cat__β-Glucan MW_Not applicable,cat__β-Glucan MW_Unspecified,num__Carrier 1 Mass Ratio,num__Carrier 2 Mass Ratio,num__β-Glucan / Carrier 2 Mass Ratio,missing__Carrier 1 Mass Ratio,missing__Carrier 2 Mass Ratio,missing__β-Glucan / Carrier 2 Mass Ratio,C1_Fragment_MolWt,C1_LogP,C1_TPSA,C1_HBD,C1_HBA,C1_RotatableBonds,C1_PEG_MW,C1_Polymer_MW,C1_Amine_Functionalized,C1_Family_Lipid,C1_Family_PCL,C1_Family_PLA,C1_Family_PLGA,C1_Family_Poloxamer,C2_Fragment_MolWt,C2_LogP,C2_TPSA,C2_HBD,C2_HBA,C2_RotatableBonds,C2_PEG_MW,C2_Polymer_MW,C2_Amine_Functionalized,C2_Family_Lipid,C2_Family_PCL,C2_Family_PLA,C2_Family_PLGA,C2_Family_Poloxamer,G_Fragment_MolWt,G_LogP,G_TPSA,G_HBD,G_HBA,G_RotatableBonds,missing__C1_Fragment_MolWt,missing__C1_LogP,missing__C1_TPSA,missing__C1_HBD,missing__C1_HBA,missing__C1_RotatableBonds,missing__C1_PEG_MW,missing__C1_Polymer_MW,missing__C2_Fragment_MolWt,missing__C2_LogP,missing__C2_TPSA,missing__C2_HBD,missing__C2_HBA,missing__C2_RotatableBonds,missing__C2_PEG_MW,missing__C2_Polymer_MW,missing__G_Fragment_MolWt,missing__G_LogP,missing__G_TPSA,missing__G_HBD,missing__G_HBA,missing__G_RotatableBonds
0,YG-SA-D1-N1-R90-10-MH-G01,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.222246,-1.222246,-1.109249,0,0,0,1.994784,1.977487,1.936862,-0.5,1.916745,1.992755,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,1.438027,1.423683,1.66603,1.964886,1.526232,1.411641,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.136868e-13,0.0,0.0,0.0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0
1,YG-SA-D1-N1-R90-10-MH-G08,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.222246,-1.222246,-0.210821,0,0,0,1.994784,1.977487,1.936862,-0.5,1.916745,1.992755,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,1.438027,1.423683,1.66603,1.964886,1.526232,1.411641,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.136868e-13,0.0,0.0,0.0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,YG-SA-D1-N1-R90-10-MH-G20,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.222246,-1.222246,1.329342,0,0,0,1.994784,1.977487,1.936862,-0.5,1.916745,1.992755,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,1.438027,1.423683,1.66603,1.964886,1.526232,1.411641,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.136868e-13,0.0,0.0,0.0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0
3,YG-SA-D1-N1-R90-10-MM-G01,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.222246,-1.222246,-1.109249,0,0,0,1.994784,1.977487,1.936862,-0.5,1.916745,1.992755,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,1.438027,1.423683,1.66603,1.964886,1.526232,1.411641,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.136868e-13,0.0,0.0,0.0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0
4,YG-SA-D1-N1-R90-10-MM-G08,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.222246,-1.222246,-0.210821,0,0,0,1.994784,1.977487,1.936862,-0.5,1.916745,1.992755,0.0,0.0,0,1.0,0.0,0.0,0.0,0.0,1.438027,1.423683,1.66603,1.964886,1.526232,1.411641,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.136868e-13,0.0,0.0,0.0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0


## Export feature tables

Feature tables are saved as CSV files with formulation IDs to support subsequent visualization, dimensionality reduction, and machine-learning workflows.

In [9]:
full_path = OUTPUT_DIR / "formulation_features_chemical_full.csv"
property_path = OUTPUT_DIR / "formulation_features_property_based.csv"
full_df.to_csv(full_path, index=False)
property_df.to_csv(property_path, index=False)
print("Saved:", full_path)
print("Saved:", property_path)

Saved: ..\data\processed\formulation_features_chemical_full.csv
Saved: ..\data\processed\formulation_features_property_based.csv


## Final quality assessment

Confirm that formulation identifiers are preserved, reference materials are matched, and exported feature matrices contain finite numerical values. Molecular descriptors and nominal material properties remain approximations; experimental particle size, PDI, zeta potential, and biological responses are excluded from predictor construction to prevent target leakage.

In [10]:
for label, table in [("Full", full_df), ("Property-based", property_df)]:
    values = table.drop(columns="ID").to_numpy(dtype=float)
    print(label, "| samples:", len(table), "| features:", values.shape[1], "| finite:", np.isfinite(values).all())
print("Reference structures:", len(profile), "| valid SMILES:", int(profile["SMILES_Valid"].sum()))
print("Experimental outputs remain in the original formulation table for later supervised modeling.")

Full | samples: 6575 | features: 86 | finite: True
Property-based | samples: 6575 | features: 74 | finite: True
Reference structures: 14 | valid SMILES: 14
Experimental outputs remain in the original formulation table for later supervised modeling.
